# 20.2 地理空间分析 / Geospatial Analysis

**中文**:很多数据自带**位置**:打车的上车点、外卖地址、门店坐标、手机信号、传感器、疫情病例。这类**地理空间数据**有它独特的一套方法——不能把经纬度当普通数字硬算(地球是球面!),而要用**地理专门的运算**:算真实距离、判断点落在哪个区域、把海量点聚合成热力图。本节从零实现地理空间分析的**三大核心操作**——大圆距离、点在多边形判定、空间网格索引——用一个模拟的**纽约打车**数据集演示。
**English**: Much data carries a **location**: ride-hailing pickups, delivery addresses, store coordinates, cell signals, sensors, disease cases. Such **geospatial data** has its own methods — you can't treat lat/lon as ordinary numbers (the Earth is a sphere!) but need **geo-specific operations**: real distances, which region a point falls in, aggregating massive points into heatmaps. This section implements the **three core operations** of geospatial analysis from scratch — great-circle distance, point-in-polygon, spatial grid indexing — on a simulated **NYC taxi** dataset.

---

**中文**:地理空间数据的三种基本几何:**点(point)**(一个坐标,如上车点)、**线(line)**(如道路、轨迹)、**多边形(polygon)**(如行政区、商圈)。坐标最常用**经纬度(WGS84)**。三大核心操作:
**English**: Three basic geometries: **points** (a coordinate, e.g. a pickup), **lines** (roads, trajectories), **polygons** (districts, business zones). Coordinates are usually **lat/lon (WGS84)**. Three core operations:

**中文**:
- **① 距离**:经纬度上的欧氏距离是**错的**(1 度经度在赤道和在高纬度对应的实际距离差很多)。要用 **Haversine(半正矢)公式**算球面上两点的**大圆距离**(沿地球表面的最短距离)。
  **Distance**: Euclidean distance on lat/lon is **wrong** (1° of longitude spans very different real distances at the equator vs high latitudes). Use the **Haversine formula** for the **great-circle distance** (shortest path along the sphere's surface).
- **② 点在多边形内(point-in-polygon)**:判断一个坐标落在哪个区域(这个上车点属于哪个行政区/计费区)。用**射线法(ray casting)**:从该点向右发一条射线,数它穿过多边形边界的次数——奇数=在内、偶数=在外。这是**空间连接(spatial join)** 的基础。
  **Point-in-polygon**: which region a coordinate falls in (which district/fare zone this pickup belongs to). Use **ray casting**: shoot a ray rightward and count boundary crossings — odd = inside, even = outside. The basis of **spatial joins**.
- **③ 空间索引/网格聚合**:海量点无法逐一处理,要把空间**切成网格单元**,把点聚合到单元里(算每格的打车量→热力图)。工业界用 **H3(Uber 的六边形网格)、Geohash、S2** 这类分层空间索引——本质都是"给地球划格子编号"。
  **Spatial indexing / grid aggregation**: massive points can't be processed one by one; **tile space into grid cells** and aggregate points per cell (trips per cell → heatmap). Industry uses hierarchical spatial indexes like **H3 (Uber's hexagons), Geohash, S2** — all "assigning grid IDs to the Earth."

> 💡 **面试速查 / Interview cheat-sheet（★★ 出行/O2O必备）**
> **中文**:地理空间三几何:点/线/面(多边形); 坐标=经纬度(WGS84)。**别用经纬度欧氏距离**——用 **Haversine 大圆距离**(球面)。**点在多边形**用射线法(奇偶穿越)→空间连接。**空间索引/网格**(H3六边形/Geohash/S2)把点聚合成格→热力图/热点检测, 是海量空间数据的可扩展性关键。工具:**GeoPandas**(pandas+几何)、**Shapely**(几何运算)、**H3/S2**(网格)、**PostGIS**(空间数据库)、**folium/kepler**(可视化)。用途:打车供需/派单、外卖配送范围、选址、物流路径、地理围栏(geofencing)、疫情热点。坑:①坐标系/投影(算面积/距离要投影到平面坐标 UTM);②经纬度顺序易搞反;③反地理编码。
> **English**: Three geometries: point/line/polygon; coordinates = lat/lon (WGS84). **Don't use Euclidean distance on lat/lon** — use the **Haversine great-circle distance** (sphere). **Point-in-polygon** via ray casting (odd/even crossings) → spatial joins. **Spatial indexing / grids** (H3 hexagons/Geohash/S2) aggregate points into cells → heatmaps / hotspot detection, key to scaling massive spatial data. Tools: **GeoPandas** (pandas + geometry), **Shapely** (geometry ops), **H3/S2** (grids), **PostGIS** (spatial DB), **folium/kepler** (viz). Uses: ride-hailing supply-demand/dispatch, delivery ranges, site selection, logistics routing, geofencing, disease hotspots. Pitfalls: ① coordinate systems/projections (project to a planar CRS like UTM for area/distance); ② lat/lon order easily swapped; ③ reverse geocoding.


In [ ]:

# ============================================================
# 模拟纽约打车数据 / simulate NYC taxi pickups
# 中文:上车点聚集在几个热点(时代广场、中央车站、金融区)。每个点=一次打车的上车经纬度。
# English: pickups cluster around hotspots (Times Sq, Grand Central, Financial District). Each point = a pickup's lat/lon.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
hotspots={"时代广场 Times Sq":(40.758,-73.985),"中央车站 Grand Central":(40.752,-73.977),
          "金融区 Financial":(40.707,-74.011),"上西区 Upper West":(40.787,-73.975)}
weights=[0.4,0.3,0.2,0.1]
lat,lon=[],[]
for (la,lo),w in zip(hotspots.values(),weights):
    k=int(6000*w)
    lat.append(rng.normal(la,0.006,k)); lon.append(rng.normal(lo,0.006,k))
lat=np.concatenate(lat); lon=np.concatenate(lon); Ntrips=len(lat)
print(f"模拟了 {Ntrips} 次打车上车点 / simulated {Ntrips} pickups around {len(hotspots)} hotspots")


**中文**:**① Haversine 大圆距离**。为什么不能用经纬度欧氏距离？下面对比 Haversine 和"把经纬度当平面坐标"的欧氏距离——在纽约的纬度,经度方向被严重高估。
**English**: **① Haversine great-circle distance**. Why not Euclidean on lat/lon? Below we compare Haversine with "treating lat/lon as planar" Euclidean — at NYC's latitude, the longitude direction is badly overestimated.


In [ ]:

# ============================================================
# ① Haversine 大圆距离 / Haversine great-circle distance
# ============================================================
def haversine(lat1, lon1, lat2, lon2):
    R=6371.0; p=np.pi/180                                    # 地球半径km, 度转弧度 / Earth radius, deg->rad
    a=(np.sin((lat2-lat1)*p/2)**2 +
       np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2)
    return 2*R*np.arcsin(np.sqrt(a))                         # 大圆距离(km)/ great-circle distance

ts=hotspots["时代广场 Times Sq"]; jfk=(40.641,-73.778)
print(f"时代广场→JFK机场 Haversine: {haversine(*ts,*jfk):.1f} km (真实约21km)")
# 对比:1度经度 vs 1度纬度 的真实距离 / real km per degree
print(f"在纽约纬度: 1° 纬度 = {haversine(40.75,-74,41.75,-74):.0f} km, 1° 经度 = {haversine(40.75,-74,40.75,-73):.0f} km")
print("→ 经度方向被压缩! 用平面欧氏距离会把东西向距离高估约 30% / lon is compressed; planar Euclidean overestimates E-W")


**中文**:**② 点在多边形内(射线法)**。定义两个"计费区"多边形,判断每个上车点落在哪个区里——这就是**空间连接**(把点和区域关联)。
**English**: **② Point-in-polygon (ray casting)**. Define two "fare zone" polygons and determine which zone each pickup falls in — this is a **spatial join** (associating points with regions).


In [ ]:

# ============================================================
# ② 点在多边形内 (射线法) / point-in-polygon (ray casting)
# ============================================================
def point_in_polygon(x, y, polygon):
    n=len(polygon); inside=False; j=n-1
    for i in range(n):
        xi,yi=polygon[i]; xj,yj=polygon[j]
        # 射线向右, 数穿过边界次数 / ray to the right, count edge crossings
        if ((yi>y)!=(yj>y)) and (x < (xj-xi)*(y-yi)/(yj-yi)+xi): inside=not inside
        j=i
    return inside
# 两个计费区(经度=x, 纬度=y)/ two fare zones (lon=x, lat=y)
midtown=[(-74.00,40.745),(-73.96,40.745),(-73.96,40.80),(-74.00,40.80)]   # 中城 / midtown
downtown=[(-74.02,40.70),(-73.99,40.70),(-73.99,40.735),(-74.02,40.735)]  # 下城 / downtown
zone=np.array(["其他" for _ in range(Ntrips)],dtype=object)
for i in range(Ntrips):
    if point_in_polygon(lon[i],lat[i],midtown): zone[i]="中城 Midtown"
    elif point_in_polygon(lon[i],lat[i],downtown): zone[i]="下城 Downtown"
from collections import Counter
print("各计费区上车量 / pickups per fare zone:", dict(Counter(zone)))


**中文**:**③ 空间网格聚合(H3/Geohash 的原理)**。把地图切成小方格,统计每格的上车量——得到**热力图**,一眼看出打车热点(供需调度、动态定价的基础)。
**English**: **③ Spatial grid aggregation (the principle of H3/Geohash)**. Tile the map into small cells and count pickups per cell — yielding a **heatmap** that reveals hotspots at a glance (the basis of supply-demand dispatch and dynamic pricing).


In [ ]:

# ============================================================
# ③ 空间网格聚合 + KNN + 可视化 / grid aggregation + KNN + visualization
# ============================================================
cell=0.004                                                  # 网格边长(度)/ cell size in degrees
lat0,lon0=40.695,-74.025                                    # 网格原点 / grid origin
gi=((lat-lat0)/cell).astype(int); gj=((lon-lon0)/cell).astype(int)   # 每点的网格坐标 / cell id
ncells=int((40.81-lat0)/cell)+1, int((-73.95-lon0)/cell)+1
grid=np.zeros(ncells)
for i,j in zip(gi,gj):
    if 0<=i<ncells[0] and 0<=j<ncells[1]: grid[i,j]+=1      # 每格计数 / count per cell
busiest=np.unravel_index(np.argmax(grid),grid.shape)
print(f"最繁忙网格上车量 / busiest cell: {int(grid.max())} 次, 非空网格数 {int((grid>0).sum())}")

# 空间 KNN:找离某点最近的 k 个上车点 / spatial KNN
query=(40.758,-73.985)                                      # 查询点(时代广场)/ query point
dists=haversine(query[0],query[1],lat,lon)
k=5; nearest=np.argsort(dists)[:k]
print(f"离时代广场最近的5个上车点距离(米)/ 5 nearest pickups (m): {(dists[nearest]*1000).round(0)}")

fig,ax=plt.subplots(1,3,figsize=(17,5))
# ① 上车点散布 + 计费区 / pickups + fare zones
cols={"中城 Midtown":"#4C72B0","下城 Downtown":"#55A868","其他":"#cccccc"}
ax[0].scatter(lon,lat,s=2,c=[cols[z] for z in zone],alpha=0.3)
for poly,name in [(midtown,"中城"),(downtown,"下城")]:
    p=np.array(poly+[poly[0]]); ax[0].plot(p[:,0],p[:,1],"k-",lw=1.5)
ax[0].set_title("上车点 + 计费区(点在多边形连接)/ pickups + fare zones"); ax[0].set_xlabel("经度 lon"); ax[0].set_ylabel("纬度 lat")
# ② 网格热力图 / grid heatmap
im=ax[1].imshow(grid,origin="lower",cmap="hot",extent=[lon0,lon0+ncells[1]*cell,lat0,lat0+ncells[0]*cell],aspect="auto")
ax[1].set_title("空间网格热力图(打车热点)/ grid heatmap"); ax[1].set_xlabel("经度"); ax[1].set_ylabel("纬度"); plt.colorbar(im,ax=ax[1],fraction=0.046,label="上车量")
# ③ KNN 查询 / KNN
ax[2].scatter(lon,lat,s=2,c="#cccccc",alpha=0.3)
ax[2].scatter(query[1],query[0],c="red",s=120,marker="*",zorder=5,label="查询点")
ax[2].scatter(lon[nearest],lat[nearest],c="#4C72B0",s=40,zorder=5,label=f"最近{k}个")
ax[2].set_title("空间 KNN(用 Haversine 距离)/ spatial KNN"); ax[2].set_xlabel("经度"); ax[2].set_ylabel("纬度"); ax[2].legend(fontsize=8)
ax[2].set_xlim(-74.0,-73.96); ax[2].set_ylim(40.745,40.775)
plt.tight_layout(); plt.savefig("/tmp/adv02_viz.png",dpi=80); plt.show()
print("三大操作:大圆距离 / 点在多边形(空间连接)/ 网格聚合(热力图)—— 地理分析的基石")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **地理数据不能当普通数字硬算**:经纬度是**球面坐标**,直接欧氏距离会把东西向距离高估(纽约纬度约 30%)。Haversine 大圆距离是正确做法。**做面积、缓冲区、精确距离时,还要把经纬度投影到平面坐标系(如 UTM)**——这是地理分析最常见的坑之一。
2. **三大操作串起整个地理分析流水线**:大圆距离(找最近的车/店)、点在多边形(把点归属到区域=空间连接)、网格聚合(把千万级点压成热力图看热点)。真实业务几乎都是这三者的组合:打车派单=KNN 找最近司机;配送范围=点在多边形;供需热力图=网格聚合;动态定价=按网格算实时供需比。
3. **诚实的工程现实**:①我们从零实现是为讲清原理,**生产中一定用成熟库**——GeoPandas(几何+pandas)、Shapely(几何运算)、H3/S2(六边形/球面网格,比方格更均匀、可分层)、PostGIS(空间数据库,带空间索引 R-tree,亿级点也能快速查询);②**空间索引是可扩展性的关键**——没有它,"找附近 1km 的点"要扫全表;③**投影选择**影响距离/面积计算的准确性;④H3 的六边形比方格好在"每个格子到邻居距离相等",更适合扩散/聚合分析。别自己造轮子,但要懂轮子的原理。

**English**:
1. **Geo data can't be treated as ordinary numbers**: lat/lon are **spherical coordinates**; naive Euclidean distance overestimates the E-W direction (~30% at NYC's latitude). Haversine is correct. **For areas, buffers, and precise distances, project lat/lon to a planar CRS (e.g. UTM)** — a very common geo pitfall.
2. **The three operations form the whole geo pipeline**: great-circle distance (find the nearest car/store), point-in-polygon (assign points to regions = spatial join), grid aggregation (compress millions of points into a hotspot heatmap). Real business is almost always a combination: ride dispatch = KNN for the nearest driver; delivery ranges = point-in-polygon; supply-demand heatmaps = grid aggregation; dynamic pricing = real-time supply/demand ratio per cell.
3. **Honest engineering reality**: ① we implement from scratch to teach the principles, but **production always uses mature libraries** — GeoPandas (geometry + pandas), Shapely (geometry ops), H3/S2 (hexagonal/spherical grids, more uniform and hierarchical than squares), PostGIS (spatial DB with R-tree spatial indexes, fast queries over billions of points); ② **spatial indexing is key to scalability** — without it, "find points within 1km" scans the whole table; ③ **projection choice** affects distance/area accuracy; ④ H3's hexagons beat squares because "each cell is equidistant to its neighbors," better for diffusion/aggregation. Don't reinvent the wheel, but understand how it works.

> 💼 **实战视角 / Practical angle**
> **中文**:地理分析是**出行(Uber/滴滴)、外卖(美团)、物流、零售选址、地图、广告 LBS** 的核心。落地栈:数据存 **PostGIS/BigQuery GIS**,处理用 **GeoPandas**,大规模网格用 **H3**(Uber 供需就用它),可视化用 **kepler.gl/folium/deck.gl**。典型任务:①**供需匹配/派单**(网格实时供需 + KNN);②**ETA 预估**(路网 + 图算法, Part 16);③**地理围栏**(点在多边形判断进出);④**选址**(热力+竞品缓冲区分析);⑤**轨迹分析**(线几何 + 地图匹配)。面试金句:*"地理数据是球面坐标, 距离用 Haversine(别用经纬度欧氏), 点归区域用点在多边形(空间连接), 海量点用 H3/Geohash 网格聚合成热力图; 生产用 GeoPandas/PostGIS/H3, 空间索引是可扩展性关键。"*
> **English**: Geo analysis is core to **ride-hailing (Uber/DiDi), delivery (Meituan), logistics, retail siting, maps, LBS ads**. Stack: store in **PostGIS/BigQuery GIS**, process with **GeoPandas**, large-scale grids with **H3** (Uber's supply-demand uses it), visualize with **kepler.gl/folium/deck.gl**. Typical tasks: ① **supply-demand matching/dispatch** (per-cell real-time supply-demand + KNN); ② **ETA prediction** (road network + graph algorithms, Part 16); ③ **geofencing** (point-in-polygon for enter/exit); ④ **site selection** (heatmap + competitor-buffer analysis); ⑤ **trajectory analysis** (line geometry + map matching). Interview line: *"Geo data is spherical; use Haversine for distance (not lat/lon Euclidean), point-in-polygon for region assignment (spatial join), and H3/Geohash grids to aggregate massive points into heatmaps; production uses GeoPandas/PostGIS/H3, and spatial indexing is key to scalability."*

---
### 小结 / Summary
- **中文**:地理数据=点/线/面 + 经纬度(球面); 三大操作:Haversine 大圆距离、点在多边形(空间连接)、网格聚合(热力图)。
- **English**: Geo data = point/line/polygon + lat/lon (spherical); three core ops: Haversine great-circle distance, point-in-polygon (spatial join), grid aggregation (heatmap).
- **中文**:别用经纬度欧氏距离; 精确面积/距离要投影(UTM); 海量点靠空间索引(H3/Geohash/R-tree)。
- **English**: Don't use Euclidean on lat/lon; project (UTM) for accurate area/distance; scale with spatial indexes (H3/Geohash/R-tree).
- **中文**:工具 GeoPandas/Shapely/H3/PostGIS; 用途:出行派单/配送/选址/地理围栏/热点。
- **English**: Tools GeoPandas/Shapely/H3/PostGIS; uses: dispatch/delivery/siting/geofencing/hotspots.
